In [3]:
# Bayesian Network WITHOUT pgmpy

import pandas as pd

# -------------------------------
# Step 1: Dataset
# -------------------------------
data = pd.DataFrame([
    {'Rain': 1, 'Sprinkler': 0, 'WetGrass': 1},
    {'Rain': 1, 'Sprinkler': 1, 'WetGrass': 1},
    {'Rain': 0, 'Sprinkler': 1, 'WetGrass': 1},
    {'Rain': 0, 'Sprinkler': 0, 'WetGrass': 0},
    {'Rain': 1, 'Sprinkler': 0, 'WetGrass': 1},
    {'Rain': 0, 'Sprinkler': 1, 'WetGrass': 1},
    {'Rain': 0, 'Sprinkler': 0, 'WetGrass': 0},
    {'Rain': 1, 'Sprinkler': 1, 'WetGrass': 1}
])

# -------------------------------
# Step 2: Compute Probabilities
# -------------------------------

# P(Rain)
p_rain = data['Rain'].value_counts(normalize=True)

# P(Sprinkler | Rain)
p_s_given_r = data.groupby('Rain')['Sprinkler'].value_counts(normalize=True)

# P(WetGrass | Rain, Sprinkler)
p_w_given_rs = data.groupby(['Rain', 'Sprinkler'])['WetGrass'].value_counts(normalize=True)

print("P(Rain):\n", p_rain)
print("\nP(Sprinkler | Rain):\n", p_s_given_r)
print("\nP(WetGrass | Rain, Sprinkler):\n", p_w_given_rs)

# -------------------------------
# Step 3: Inference
# Compute P(Rain | WetGrass = 1)
# -------------------------------

def get_prob_w_given_r(r):
    total = 0
    for s in [0, 1]:
        try:
            p_s_r = p_s_given_r[r][s]
        except:
            p_s_r = 0

        try:
            p_w_rs = p_w_given_rs[(r, s)][1]
        except:
            p_w_rs = 0

        total += p_s_r * p_w_rs
    return total

# Bayes formula
p_w = 0
for r in [0, 1]:
    p_w += p_rain[r] * get_prob_w_given_r(r)

posterior = {}
for r in [0, 1]:
    posterior[r] = (p_rain[r] * get_prob_w_given_r(r)) / p_w

# -------------------------------
# Step 4: Output
# -------------------------------
print("\nP(Rain | WetGrass=1):")
for k, v in posterior.items():
    print(f"Rain={k} -> {v:.4f}")

P(Rain):
 Rain
1    0.5
0    0.5
Name: proportion, dtype: float64

P(Sprinkler | Rain):
 Rain  Sprinkler
0     0            0.5
      1            0.5
1     0            0.5
      1            0.5
Name: proportion, dtype: float64

P(WetGrass | Rain, Sprinkler):
 Rain  Sprinkler  WetGrass
0     0          0           1.0
      1          1           1.0
1     0          1           1.0
      1          1           1.0
Name: proportion, dtype: float64

P(Rain | WetGrass=1):
Rain=0 -> 0.3333
Rain=1 -> 0.6667


In [4]:
import pandas as pd

# Dataset
data = pd.DataFrame({
    'Rain': [0,0,1,1,0,1,1,0],
    'TrafficJam': [1,1,1,0,1,0,0,0],
    'ArriveLate': [1,1,1,0,0,1,1,1]
})

print("Dataset:\n", data)

# Step 1: P(Rain)
p_rain = data['Rain'].value_counts(normalize=True)
print("\nP(Rain):\n", p_rain)

# Step 2: P(TrafficJam | Rain)
p_traffic_given_rain = pd.crosstab(data['TrafficJam'], data['Rain'], normalize='columns')
print("\nP(TrafficJam | Rain):\n", p_traffic_given_rain)

# Step 3: P(ArriveLate | TrafficJam)
p_late_given_traffic = pd.crosstab(data['ArriveLate'], data['TrafficJam'], normalize='columns')
print("\nP(ArriveLate | TrafficJam):\n", p_late_given_traffic)

# Step 4: Query → P(ArriveLate | Rain = 1)

# Use total probability:
# P(Late | Rain) = Σ P(Late | TrafficJam) * P(TrafficJam | Rain)

p_late_given_rain = (
    p_late_given_traffic[1][1] * p_traffic_given_rain[1][1] +   # Traffic=1
    p_late_given_traffic[1][0] * p_traffic_given_rain[0][1]     # Traffic=0
)

p_not_late_given_rain = (
    p_late_given_traffic[0][1] * p_traffic_given_rain[1][1] +
    p_late_given_traffic[0][0] * p_traffic_given_rain[0][1]
)

print("\nFinal Result:")
print("P(ArriveLate=1 | Rain=1):", round(p_late_given_rain, 3))
print("P(ArriveLate=0 | Rain=1):", round(p_not_late_given_rain, 3))


Dataset:
    Rain  TrafficJam  ArriveLate
0     0           1           1
1     0           1           1
2     1           1           1
3     1           0           0
4     0           1           0
5     1           0           1
6     1           0           1
7     0           0           1

P(Rain):
 Rain
0    0.5
1    0.5
Name: proportion, dtype: float64

P(TrafficJam | Rain):
 Rain           0     1
TrafficJam            
0           0.25  0.75
1           0.75  0.25

P(ArriveLate | TrafficJam):
 TrafficJam     0     1
ArriveLate            
0           0.25  0.25
1           0.75  0.75

Final Result:
P(ArriveLate=1 | Rain=1): 0.375
P(ArriveLate=0 | Rain=1): 0.375


In [7]:
import pandas as pd

data = pd.DataFrame({
    'Disease': [1,0,1,0,1,0,1,0],
    'Fever':   [1,0,1,0,1,0,0,0],
    'Cough':   [1,0,1,1,0,0,1,0]
})

# P(Disease)
p_disease = data['Disease'].value_counts(normalize=True)

# P(Fever | Disease)
p_fever_given_disease = pd.crosstab(data['Fever'], data['Disease'], normalize='columns')

# P(Cough | Disease)
p_cough_given_disease = pd.crosstab(data['Cough'], data['Disease'], normalize='columns')

# Query: P(Disease=1 | Fever=1, Cough=1)
num = p_disease[1] * p_fever_given_disease[1][1] * p_cough_given_disease[1][1]
den = num + (p_disease[0] * p_fever_given_disease[1][0] * p_cough_given_disease[1][0])

print("P(Disease=1 | Fever=1, Cough=1):", round(num/den, 3))


P(Disease=1 | Fever=1, Cough=1): 0.9


In [8]:
import pandas as pd

# Dataset
data = pd.DataFrame({
    'Spam': [1,0,1,0,1,0,1,0],
    'Offer': [1,0,1,1,1,0,0,0],
    'UnknownSender': [1,0,1,1,0,0,1,0]
})

print("Dataset:\n", data)

# Step 1: P(Spam)
p_spam = data['Spam'].value_counts(normalize=True)
print("\nP(Spam):\n", p_spam)

# Step 2: P(Offer | Spam)
p_offer_given_spam = pd.crosstab(data['Offer'], data['Spam'], normalize='columns')
print("\nP(Offer | Spam):\n", p_offer_given_spam)

# Step 3: P(UnknownSender | Spam)
p_unknown_given_spam = pd.crosstab(data['UnknownSender'], data['Spam'], normalize='columns')
print("\nP(UnknownSender | Spam):\n", p_unknown_given_spam)

# Step 4: Apply Bayes Theorem
# Query: Offer=1, UnknownSender=1

num = p_spam[1] * p_offer_given_spam[1][1] * p_unknown_given_spam[1][1]
den = num + (p_spam[0] * p_offer_given_spam[1][0] * p_unknown_given_spam[1][0])

print("\nFinal Result:")
print("P(Spam=1 | Offer=1, UnknownSender=1):", round(num/den, 3))


Dataset:
    Spam  Offer  UnknownSender
0     1      1              1
1     0      0              0
2     1      1              1
3     0      1              1
4     1      1              0
5     0      0              0
6     1      0              1
7     0      0              0

P(Spam):
 Spam
1    0.5
0    0.5
Name: proportion, dtype: float64

P(Offer | Spam):
 Spam      0     1
Offer            
0      0.75  0.25
1      0.25  0.75

P(UnknownSender | Spam):
 Spam              0     1
UnknownSender            
0              0.75  0.25
1              0.25  0.75

Final Result:
P(Spam=1 | Offer=1, UnknownSender=1): 0.9
